# Bài thực hành 2 - CNN

Notebook chạy toàn bộ 4 bài trên Google Colab (GPU NVIDIA L4).

## Kiểm tra GPU

In [1]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU - đổi runtime sang GPU!")

Fri Oct 23 14:38:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Lấy code từ Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, shutil, zipfile

DRIVE = "/content/drive/MyDrive"
REPO = "/content/bai-thuc-hanh-2"
CKPT_DRIVE = "/content/drive/MyDrive/BTH2_checkpoints"
TMP = "/content/_code_tmp"

def code_root(base):
    """Thư mục chứa assignment-2.py bên trong base (None nếu không có)."""
    hits = [h for h in glob.glob(os.path.join(base, "**", "assignment-2.py"), recursive=True) if "__MACOSX" not in h]
    return os.path.dirname(min(hits, key=len)) if hits else None

def candidates(max_depth=3):
    """Các thư mục / file zip trong My Drive (tối đa 3 cấp), bỏ qua thư mục checkpoint."""
    for depth in range(1, max_depth + 1):
        for path in sorted(glob.glob(os.path.join(DRIVE, *["*"] * depth))):
            if "BTH2_checkpoints" not in path:
                yield path

src = None
for path in candidates():
    # Trường hợp 1: upload cả thư mục code lên Drive
    if os.path.basename(path) == "assignment-2.py":
        src = os.path.dirname(path)
        print("Tìm thấy thư mục code:", src)
        break
    # Trường hợp 2: upload file zip (tên bất kỳ, vd "bai-thuc-hanh-2 (1).zip")
    if path.lower().endswith(".zip"):
        try:
            with zipfile.ZipFile(path) as zf:
                if not any(n.endswith("assignment-2.py") and "__MACOSX" not in n for n in zf.namelist()):
                    continue
                print("Tìm thấy file zip:", path)
                shutil.rmtree(TMP, ignore_errors=True)
                zf.extractall(TMP)
            src = code_root(TMP)
            break
        except zipfile.BadZipFile:
            continue

assert src is not None, "Không tìm thấy code (assignment-2.py) trong My Drive. Kiểm tra lại file đã upload."

# Chép code vào REPO (không đụng tới dataset/ nếu đã có)
os.makedirs(REPO, exist_ok=True)
for item in os.listdir(src):
    if item in ("checkpoints", "dataset", "__MACOSX"):
        continue
    s, d = os.path.join(src, item), os.path.join(REPO, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

# checkpoints/ trong repo trỏ vào Drive để không mất kết quả khi bị ngắt kết nối
os.makedirs(CKPT_DRIVE, exist_ok=True)
link = os.path.join(REPO, "checkpoints")
if os.path.islink(link) or os.path.isfile(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
os.symlink(CKPT_DRIVE, link)

assert os.path.exists(os.path.join(REPO, "assignment-1.py")), "Thiếu assignment-1.py sau khi chép code."
print("Code đã sẵn sàng:", sorted(os.listdir(REPO)))

Mounted at /content/drive
Tìm thấy file zip: /content/drive/MyDrive/bai-thuc-hanh-2.zip
Code đã sẵn sàng: ['.gitignore', '24521715.txt', 'README.md', 'assignment-1.py', 'assignment-2.py', 'assignment-3.py', 'assignment-4.py', 'checkpoints', 'data_utils', 'models', 'requirements.txt', 'utils']


## Cài thư viện

In [3]:
!pip install -q idx2numpy gdown

  Preparing metadata (setup.py) ... done


## Tải dữ liệu VinaFood21

In [4]:
DATA_ZIP_ON_DRIVE = ""   # vd: "/content/drive/MyDrive/VinaFood21.zip" nếu gdown lỗi; để trống thì tải bằng gdown
RAW = "/content/raw_data"

import os
os.makedirs(RAW, exist_ok=True)
if DATA_ZIP_ON_DRIVE:
    !cp "{DATA_ZIP_ON_DRIVE}" /content/data.zip
elif not os.path.exists("/content/data.zip"):
    !gdown 1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR -O /content/data.zip
!unzip -q -o /content/data.zip -d {RAW}

# Nếu trong zip còn zip con thì giải nén tiếp
import glob
for inner in glob.glob(f"{RAW}/**/*.zip", recursive=True):
    print("Giải nén thêm:", inner)
    !unzip -q -o "{inner}" -d "{os.path.dirname(inner)}"
!find {RAW} -maxdepth 3 -type d | head -40

Downloading...
From (original): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR
From (redirected): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR&confirm=t&uuid=339aefd5-dc80-4039-bb51-48fb1dc5a196
To: /content/data.zip
100% 2.71G/2.71G [00:21<00:00, 125MB/s]
/content/raw_data
/content/raw_data/VinaFood21
/content/raw_data/VinaFood21/test
/content/raw_data/VinaFood21/test/bap-nuong
/content/raw_data/VinaFood21/test/ha-cao
/content/raw_data/VinaFood21/test/bo-kho
/content/raw_data/VinaFood21/test/bot-chien
/content/raw_data/VinaFood21/test/bo-la-lot
/content/raw_data/VinaFood21/test/goi-ga
/content/raw_data/VinaFood21/test/ca-ri
/content/raw_data/VinaFood21/test/hoanh-thanh-nuoc
/content/raw_data/VinaFood21/test/banh-mi-chao
/content/raw_data/VinaFood21/test/canh-kho-qua
/content/raw_data/VinaFood21/test/banh-u
/content/raw_data/VinaFood21/test/canh-khoai-mo
/content/raw_data/VinaFood21/test/banh-uot
/content/raw_data/VinaFood21/test/banh-hoi
/content/

## Thư mục train / test

In [5]:
import os

def find_split_dirs(base):
    """Tìm thư mục có cả train và test (không phân biệt hoa thường), mỗi cái chứa các thư mục lớp."""
    for dirpath, dirnames, _ in os.walk(base):
        lower = {d.lower(): d for d in dirnames}
        if "train" in lower and "test" in lower:
            train_dir = os.path.join(dirpath, lower["train"])
            test_dir = os.path.join(dirpath, lower["test"])
            n_classes = len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
            if n_classes >= 2:
                return train_dir, test_dir
    raise FileNotFoundError("Không tìm thấy cặp thư mục train/test - xem kết quả `find` ở ô trên và sửa tay DATA_DIR.")

train_dir, test_dir = find_split_dirs(RAW)

# Chuẩn hoá về DATA_DIR/train và DATA_DIR/test như code yêu cầu
DATA_DIR = "/content/VinaFood21"
os.makedirs(DATA_DIR, exist_ok=True)
for name, target in [("train", train_dir), ("test", test_dir)]:
    link = os.path.join(DATA_DIR, name)
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(target, link)

for split in ["train", "test"]:
    root = os.path.join(DATA_DIR, split)
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    total = sum(len(os.listdir(os.path.join(root, c))) for c in classes)
    print(f"{split}: {len(classes)} lớp, {total} ảnh")
print("Lớp:", classes)

train: 21 lớp, 10044 ảnh
test: 21 lớp, 6682 ảnh
Lớp: ['banh-can', 'banh-hoi', 'banh-mi-chao', 'banh-tet', 'banh-trang-tron', 'banh-u', 'banh-uot', 'bap-nuong', 'bo-kho', 'bo-la-lot', 'bot-chien', 'ca-ri', 'canh-kho-qua', 'canh-khoai-mo', 'ga-nuong', 'goi-ga', 'ha-cao', 'hoanh-thanh-nuoc', 'pha-lau', 'tau-hu', 'thit-kho-trung']


## Dữ liệu MNIST (Bài 1)

In [6]:
import os, glob, gzip, shutil

MNIST_DIR = f"{REPO}/dataset/MNIST"
os.makedirs(MNIST_DIR, exist_ok=True)
NAMES = ["train-images-idx3-ubyte", "train-labels-idx1-ubyte", "t10k-images-idx3-ubyte", "t10k-labels-idx1-ubyte"]

def find_file(name):
    # chấp nhận cả dạng train-images.idx3-ubyte và bản nén .gz
    variants = {name, name.replace("-idx", ".idx")}
    for path in glob.glob(f"{RAW}/**/*", recursive=True):
        base = os.path.basename(path)
        for v in variants:
            if base == v or base == v + ".gz":
                return path
    return None

found = {n: find_file(n) for n in NAMES}
if not all(found.values()):
    print("Không có MNIST trong dữ liệu đã tải -> tải bằng torchvision")
    from torchvision.datasets import MNIST
    MNIST("/content/mnist_tv", train=True, download=True)
    MNIST("/content/mnist_tv", train=False, download=True)
    found = {n: f"/content/mnist_tv/MNIST/raw/{n}" for n in NAMES}

for name, src in found.items():
    dst = os.path.join(MNIST_DIR, name)
    if src.endswith(".gz"):
        with gzip.open(src, "rb") as fi, open(dst, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    else:
        shutil.copy(src, dst)
!ls -la {MNIST_DIR}

Không có MNIST trong dữ liệu đã tải -> tải bằng torchvision


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.76MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.91MB/s]


total 53680
drwxr-xr-x 2 root root     4096 Oct 23 14:40 .
drwxr-xr-x 3 root root     4096 Oct 23 14:40 ..
-rw-r--r-- 1 root root  7840016 Oct 23 14:40 t10k-images-idx3-ubyte
-rw-r--r-- 1 root root    10008 Oct 23 14:40 t10k-labels-idx1-ubyte
-rw-r--r-- 1 root root 47040016 Oct 23 14:40 train-images-idx3-ubyte
-rw-r--r-- 1 root root    60008 Oct 23 14:40 train-labels-idx1-ubyte


## Huấn luyện và đánh giá

In [7]:
# Bài 1: LeNet - MNIST
!cd {REPO} && python assignment-1.py

Epoch 1 - Training: 100% 1875/1875 [00:07<00:00, 240.04it/s, loss=0.867]
Evaluating: 100% 10000/10000 [00:05<00:00, 1676.51it/s]
	- accuracy: 0.9148
	- precision: 0.9147890534612249
	- recall: 0.9135228050441035
	- f1: 0.9137344889906469
Epoch 2 - Training: 100% 1875/1875 [00:05<00:00, 367.35it/s, loss=0.222]
Evaluating: 100% 10000/10000 [00:05<00:00, 1695.91it/s]
	- accuracy: 0.9522
	- precision: 0.9516958402151114
	- recall: 0.9519659714905337
	- f1: 0.9517380395454277
Epoch 3 - Training: 100% 1875/1875 [00:05<00:00, 368.96it/s, loss=0.145]
Evaluating: 100% 10000/10000 [00:05<00:00, 1705.64it/s]
	- accuracy: 0.9633
	- precision: 0.9632491809113508
	- recall: 0.9629885280102508
	- f1: 0.9630011212221022
Epoch 4 - Training: 100% 1875/1875 [00:05<00:00, 374.73it/s, loss=0.11]
Evaluating: 100% 10000/10000 [00:05<00:00, 1682.32it/s]
	- accuracy: 0.974
	- precision: 0.973889536109953
	- recall: 0.9737748013811258
	- f1: 0.9737962090320622
Epoch 5 - Training: 100% 1875/1875 [00:05<00:00, 37

In [8]:
# Bài 2: GoogLeNet - VinaFood21
!cd {REPO} && python assignment-2.py --data_dir {DATA_DIR} --num_workers 4

21 lớp | train 9049 | dev 995 | test 6682
Epoch 1 - Training:  66% 188/283 [00:26<00:12,  7.41it/s, loss=3]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1 - Training: 100% 283/283 [00:39<00:00,  7.14it/s, loss=2.99]
Epoch 1 - Dev: 100% 32/32 [00:04<00:00,  6.60it/s]
Epoch 1: train loss = 2.9903
	- dev accuracy: 0.0844
	- dev precision: 0.0040
	- dev recall: 0.0476
	- dev f1: 0.0074
	=> Lưu checkpoint mới (dev f1 = 0.0074)
Epoch 2 - Training:  13% 38/283 [00:05<00:34,  7.11it/s, loss=2.99]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2 - Training: 100% 283/283 [00:38<00:00,  7.38it/s, loss=2.94]
Epoch 2 - Dev: 100% 32/32 [00:04<00:00,  6.89it/s]
Epoch 2: train loss = 2.9358
	- dev accuracy: 0.1126
	- dev precision: 0.02

In [9]:
# Bài 3: ResNet-18 - VinaFood21
!cd {REPO} && python assignment-3.py --data_dir {DATA_DIR} --num_workers 4

21 lớp | train 9049 | dev 995 | test 6682
Epoch 1 - Training:  49% 138/283 [00:19<00:18,  7.64it/s, loss=2.79]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1 - Training: 100% 283/283 [00:38<00:00,  7.29it/s, loss=2.67]
Epoch 1 - Dev: 100% 32/32 [00:04<00:00,  6.77it/s]
Epoch 1: train loss = 2.6714
	- dev accuracy: 0.2332
	- dev precision: 0.1544
	- dev recall: 0.1782
	- dev f1: 0.1502
	=> Lưu checkpoint mới (dev f1 = 0.1502)
Epoch 2 - Training:  33% 93/283 [00:12<00:25,  7.47it/s, loss=2.47]/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2 - Training: 100% 283/283 [00:38<00:00,  7.26it/s, loss=2.34]
Epoch 2 - Dev: 100% 32/32 [00:04<00:00,  6.67it/s]
Epoch 2: train loss = 2.3416
	- dev accuracy: 0.2734
	- dev precision: 0

In [10]:
# Bài 4: Fine-tune ResNet50 pretrained (HuggingFace) - VinaFood21
!cd {REPO} && python assignment-4.py --data_dir {DATA_DIR} --num_workers 4

preprocessor_config.json: 100% 266/266 [00:00<00:00, 1.16MB/s]
21 lớp | train 9049 | dev 995 | test 6682
config.json: 100% 69.6k/69.6k [00:00<00:00, 76.5MB/s]
[transformers] You passed `num_labels=21` which is incompatible to the `id2label` map of length `1000`.

model.safetensors: downloading bytes:  81% 83.3M/102M [00:01<00:00, 89.7MB/s, 4.00MB/s  ]
model.safetensors: downloading bytes: 100% 96.1M/96.1M [00:02<00:00, 43.7MB/s, 9.06MB/s  ]
model.safetensors: reconstructing file: 100% 102M/102M [00:02<00:00, 46.6MB/s, 9.73MB/s  ]
Loading weights: 100% 320/320 [00:00<00:00, 4174.57it/s]
[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                             
--------------------+----------+---------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torc

## Tổng hợp kết quả

In [11]:
import json, os
import pandas as pd

rows = []
for i, name in [(2, "GoogLeNet"), (3, "ResNet-18"), (4, "ResNet50 (pretrained)")]:
    path = f"{CKPT_DRIVE}/assignment-{i}/results.json"
    if os.path.exists(path):
        test = json.load(open(path))["test"]
        rows.append({"Bài": i, "Mô hình": name, **{k: round(v, 4) for k, v in test.items()}})
pd.DataFrame(rows)

,Bài,Mô hình,accuracy,precision,recall,f1
0,2,GoogLeNet,0.3957,0.3814,0.3519,0.3443
1,3,ResNet-18,0.6152,0.6431,0.6001,0.6057
2,4,ResNet50 (pretrained),0.8493,0.8481,0.8535,0.8497


In [12]:
# Báo cáo theo từng lớp của một bài (đổi số bài nếu cần)
print(open(f"{CKPT_DRIVE}/assignment-4/classification_report.txt").read())

                  precision    recall  f1-score   support

        banh-can     0.8302    0.9129    0.8696       241
        banh-hoi     0.8307    0.8814    0.8553       607
    banh-mi-chao     0.8852    0.9343    0.9091       289
        banh-tet     0.9643    0.8159    0.8839       364
 banh-trang-tron     0.8734    0.8932    0.8832       309
          banh-u     0.8173    0.9148    0.8633       176
        banh-uot     0.8480    0.7571    0.8000       560
       bap-nuong     0.9720    0.9720    0.9720       143
          bo-kho     0.7963    0.7633    0.7795       338
       bo-la-lot     0.8987    0.9105    0.9046       458
       bot-chien     0.8212    0.8152    0.8182       276
           ca-ri     0.5337    0.6491    0.5858       171
    canh-kho-qua     0.9061    0.9655    0.9349       290
   canh-khoai-mo     0.9179    0.9040    0.9109       198
        ga-nuong     0.8257    0.8280    0.8268       372
          goi-ga     0.8067    0.7483    0.7764       290
          ha-